In [ ]:
## Getting Started with PyTorch
# https://www.kubeflow.org/docs/components/trainer/getting-started/

In [1]:
# Install dependencies
!pip install -U kubeflow torch --quiet

In [2]:
# ==================== IMPORTS ====================
import os
import sys
import time
import kubeflow.trainer
from kubeflow.trainer import KubernetesBackendConfig, TrainerClient, CustomTrainer

In [3]:
def check_gpu_availability():
    """Check if GPU resources are available"""
    import subprocess
    try:
        result = subprocess.run(
            ["kubectl", "get", "nodes", "-o", "json"],
            capture_output=True,
            text=True
        )
        if "nvidia.com/gpu" in result.stdout:
            print("✅ GPU nodes detected")
            return True
        else:
            print("⚠️  No GPU nodes detected, falling back to CPU")
            # Remove GPU from resources
            TrainingConfig.RESOURCES_PER_NODE.pop("nvidia.com/gpu", None)
            TrainingConfig.RESOURCE_LIMITS.pop("nvidia.com/gpu", None)
            return False
    except:
        print("⚠️  Could not check GPU availability")
        return False


# Check environment
print("🔍 Checking training environment...")
check_gpu_availability()

🔍 Checking training environment...
⚠️  No GPU nodes detected, falling back to CPU
⚠️  Could not check GPU availability


False

In [ ]:
# ==================== CONFIGURATION ====================
class TrainingConfig:
    """Centralized configuration for distributed training"""
    
    # Cluster configuration
    NUM_NODES = 3
    NODE_SELECTOR = {
        "node-type": "gpu-node"  # Optional: Select specific node types
    }
    
    # Resource allocation per pod
    RESOURCES_PER_NODE = {
        "nvidia.com/gpu": 1,      # 2 GPUs per node
        "cpu": "4",               # 4 CPUs per node
        "memory": "8Gi"           # 8GB memory per node
    }
    
    # Optional: Resource limits (prevent pods from using too much)
    RESOURCE_LIMITS = {
        "cpu": "8",
        "memory": "16Gi",
        "nvidia.com/gpu": "2"
    }
    
    # Timeout configurations
    STARTUP_TIMEOUT = 300  # 5 minutes
    TRAINING_TIMEOUT = 3600  # 1 hour


In [ ]:
# ==================== DISTRIBUTED TRAINING FUNCTION ====================
def distributed_training_job():
    """Main training function that runs on each worker"""
    import torch
    import torch.distributed as dist
    import torch.nn as nn
    import torch.optim as optim
    from torch.nn.parallel import DistributedDataParallel as DDP
    
    # Initialize distributed process group
    rank = int(os.environ.get('RANK', 0))
    local_rank = int(os.environ.get('LOCAL_RANK', 0))
    world_size = int(os.environ.get('WORLD_SIZE', 1))
    
    # Set device
    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
        device = torch.device(f"cuda:{local_rank}")
        backend = "nccl"
    else:
        device = torch.device("cpu")
        backend = "gloo"
    
    # Initialize process group
    dist.init_process_group(
        backend=backend,
        init_method="env://",
        world_size=world_size,
        rank=rank
    )
    
    try:
        # Log distributed environment info
        if rank == 0:
            print("=" * 50)
            print("PYTORCH DISTRIBUTED TRAINING INITIALIZED")
            print(f"Backend: {backend}")
            print(f"World Size: {world_size}")
            print(f"Using device: {device}")
            print(f"PyTorch version: {torch.__version__}")
            print(f"CUDA available: {torch.cuda.is_available()}")
            if torch.cuda.is_available():
                print(f"GPU count: {torch.cuda.device_count()}")
                print(f"GPU name: {torch.cuda.get_device_name(local_rank)}")
            print("=" * 50)
        
        # Example: Simple training loop with DDP
        if rank == 0:
            print("Setting up model and optimizer...")
        
        # Create a simple model
        model = nn.Sequential(
            nn.Linear(10, 50),
            nn.ReLU(),
            nn.Linear(50, 1)
        ).to(device)
        
        # Wrap model with DDP
        model = DDP(model, device_ids=[local_rank] if torch.cuda.is_available() else None)
        
        # Setup optimizer
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()
        
        # Simulate training
        if rank == 0:
            print("Starting training loop...")
        
        for epoch in range(3):  # Reduced epochs for demonstration
            # Simulate data
            data = torch.randn(32, 10).to(device)
            target = torch.randn(32, 1).to(device)
            
            # Training step
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            # Sync and log
            dist.all_reduce(loss, op=dist.ReduceOp.AVG)
            
            if rank == 0:
                print(f"Epoch [{epoch + 1}/3], Loss: {loss.item():.4f}")
        
        if rank == 0:
            print("Training completed successfully!")
            
    except Exception as e:
        print(f"Rank {rank} encountered error: {str(e)}")
        raise
    finally:
        # Cleanup
        dist.destroy_process_group()


In [ ]:
# ==================== JOB MANAGEMENT ====================
def monitor_job(job_id, follow_logs=True):
    """Monitor job status and logs"""
    trainer = TrainerClient()
    job = trainer.get_job(name=job_id)
    
    print(f"\nJob ID: {job_id}")
    print(f"Job Status: {job.status}")
    print(f"Creation Time: {job.creation_timestamp}")
    
    # Display step information
    print("\nJob Steps:")
    for step in job.steps:
        print(f"  - {step.name}: {step.status}, "
              f"Devices: {step.device} x {step.device_count}")
    
    # Follow logs if requested
    if follow_logs and job.status in ["RUNNING", "PENDING"]:
        print("\n=== TRAINING LOGS (Master Node) ===")
        try:
            for log_line in trainer.get_job_logs(job_id, follow=True):
                print(log_line)
        except KeyboardInterrupt:
            print("\nLog monitoring stopped by user")
    
    return job.status

In [ ]:
# ==================== MAIN EXECUTION ====================
print("Initializing Kubeflow Trainer...")

# Initialize backend configuration
backend_config = KubernetesBackendConfig(
    namespace="kubeflow-m-xochicale"  # Specify your namespace or just leave it blank as backend_config = KubernetesBackendConfig()
)
trainer = TrainerClient(backend_config=backend_config)

# List available runtimes
print("\nAvailable Runtimes:")
for runtime in trainer.list_runtimes():
    print(f"  - {runtime.name}")


# Create training job
print("\nCreating distributed training job...")

job_id = trainer.train(
    runtime=trainer.get_runtime("torch-distributed"),
    trainer=CustomTrainer(
        func=distributed_training_job,
        num_nodes=TrainingConfig.NUM_NODES,
        resources_per_node=TrainingConfig.RESOURCES_PER_NODE,
    ),
)


In [ ]:
print(f"\n✅ Job submitted successfully! Job ID: {job_id}")

# Wait a moment for job to start
print("\n⏳ Waiting for job to initialize...")
time.sleep(10)

# Monitor job
status = monitor_job(job_id, follow_logs=True)

print(f"\n🎯 Job finished with status: {status}")
